# FraudeCero · Grupo 6 · 01 · Limpieza (capa PLATA)
**Proyecto Integrador Big Data – Uniciencia 2026-II** · Integrantes: Juan David Montoya · William Conde Villamizar


**Objetivo:** transformar las tablas bronce en tablas **plata** limpias, tipificadas y unificadas. Cada transformación registra **filas antes y después** en una bitácora (A2.1).

| # | Transformación | Tabla |
|---|---|---|
| T1 | Deduplicación | transacciones (F1) y sfc_tarjetas (F2) |
| T2 | Descarte de nulos | transacciones y sfc_tarjetas |
| T3 | Tipificación y variables derivadas | transacciones (hora, día, log del monto) y sfc_tarjetas (fecha, números) |
| T4 | Estandarización de categorías | sfc_tarjetas (entidad, franquicia, indicador) |
| T5 | Filtrado de atípicos | transacciones (montos inválidos + marca IQR) y sfc_tarjetas (valores negativos) |
| T6 | Unión de fuentes | plata.exposicion = transacciones SFC × tasa de fraude observada |

## Configuración común
Esta celda es igual en todos los notebooks. Detecta si se ejecuta en **Google Colab** o en **Jupyter / GitHub Actions**, instala PySpark 3.5 + Delta Lake (OSS) y define:
- `LAKE`: carpeta del Lakehouse (capas `bronce`, `plata`, `oro`).
- `EVID`: carpeta `99_evidencias` (tablas de evidencia, log de monitoreo, tableros HTML).
- `guardar()` / `leer()`: escriben y leen tablas **Delta** (si Delta no está disponible usan Parquet y lo avisan).
- `registrar()`: escribe en el log de monitoreo el tiempo, las filas, el % de nulos y la memoria de cada etapa.

> En Colab cada notebook corre en una máquina distinta, por eso el Lakehouse se respalda en Google Drive al final de cada notebook y se restaura al inicio del siguiente.

In [ ]:
# ===================== CONFIGURACIÓN COMÚN FRAUDECERO =====================
NOTEBOOK = "01_limpieza"
import os, sys, time, json, shutil, datetime, subprocess, hashlib
EN_COLAB = "google.colab" in sys.modules

if EN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "pyspark==3.5.3", "delta-spark==3.2.1", "kagglehub", "psutil", "plotly"], check=True)
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DRIVE = "/content/drive/MyDrive/FraudeCero"
    LAKE = "/content/lakehouse"                      # disco local (rápido)
    LAKE_RESPALDO = f"{BASE_DRIVE}/lakehouse"        # respaldo persistente en Drive
    EVID = f"{BASE_DRIVE}/99_evidencias"
else:
    RAIZ = os.environ.get("FRAUDECERO_RAIZ", os.path.abspath(".."))
    LAKE = os.environ.get("FRAUDECERO_LAKE", os.path.join(RAIZ, "lakehouse"))
    LAKE_RESPALDO = None
    EVID = os.environ.get("FRAUDECERO_EVID", os.path.join(RAIZ, "99_evidencias"))

for carpeta in [LAKE, EVID, f"{EVID}/monitoreo", f"{EVID}/tablas"]:
    os.makedirs(carpeta, exist_ok=True)

def restaurar_lake():
    """En Colab: trae el Lakehouse desde Drive si el disco local está vacío."""
    if LAKE_RESPALDO and os.path.exists(LAKE_RESPALDO) and not os.listdir(LAKE):
        shutil.copytree(LAKE_RESPALDO, LAKE, dirs_exist_ok=True)
        print("Lakehouse restaurado desde Drive")

def respaldar_lake():
    """En Colab: copia el Lakehouse a Drive (sin la zona landing de archivos crudos)."""
    if LAKE_RESPALDO:
        shutil.copytree(LAKE, LAKE_RESPALDO, dirs_exist_ok=True,
                        ignore=shutil.ignore_patterns("landing"))
        print("Lakehouse respaldado en Drive:", LAKE_RESPALDO)

restaurar_lake()

from pyspark.sql import SparkSession, functions as F, types as T
from pyspark.sql.window import Window

FORMATO = "delta"
try:
    from delta import configure_spark_with_delta_pip
    builder = (SparkSession.builder.appName(f"FraudeCero-{NOTEBOOK}")
               .config("spark.driver.memory", "6g")
               .config("spark.sql.shuffle.partitions", "8")
               .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
               .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog"))
    spark = configure_spark_with_delta_pip(builder).getOrCreate()
    spark.range(1).write.format("delta").mode("overwrite").save(f"{LAKE}/_prueba_delta")
except Exception as e:
    print("⚠️ Delta Lake no disponible, se usará Parquet. Detalle:", str(e)[:300])
    FORMATO = "parquet"
    spark = (SparkSession.builder.appName(f"FraudeCero-{NOTEBOOK}")
             .config("spark.driver.memory", "6g")
             .config("spark.sql.shuffle.partitions", "8").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

def ruta(capa, tabla):
    return f"{LAKE}/{capa}/{tabla}"

def leer(capa, tabla):
    df = spark.read.format(FORMATO).load(ruta(capa, tabla))
    df.createOrReplaceTempView(f"{capa}_{tabla}")
    return df

def guardar(df, capa, tabla, particion=None):
    w = df.write.format(FORMATO).mode("overwrite").option("overwriteSchema", "true")
    if particion:
        w = w.partitionBy(particion)
    w.save(ruta(capa, tabla))
    return leer(capa, tabla)

# ---------- Monitoreo: tiempo, filas, % nulos y memoria por etapa ----------
import psutil, pandas as pd
LOG_MON = f"{EVID}/monitoreo/log_ejecuciones.csv"
ARCHIVO_RUN = f"{EVID}/monitoreo/run_actual.txt"
RUN_ID = os.environ.get("FRAUDECERO_RUN_ID")
if not RUN_ID:
    if NOTEBOOK == "00_ingesta" or not os.path.exists(ARCHIVO_RUN):
        RUN_ID = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")   # cada corrida del pipeline empieza en 00
        with open(ARCHIVO_RUN, "w") as f:
            f.write(RUN_ID)
    else:
        RUN_ID = open(ARCHIVO_RUN).read().strip()
ORIGEN = "github_actions" if os.environ.get("GITHUB_ACTIONS") else ("colab" if EN_COLAB else "jupyter")

def memoria_mb():
    py = psutil.Process(os.getpid()).memory_info().rss / 1024**2
    try:
        rt = spark.sparkContext._jvm.java.lang.Runtime.getRuntime()
        jvm = (rt.totalMemory() - rt.freeMemory()) / 1024**2
    except Exception:
        jvm = 0.0
    return round(py + jvm, 1)

def registrar(etapa, t0, df=None, filas=None):
    """Registra una etapa en el log de monitoreo. t0 = time.time() al inicio de la etapa."""
    duracion = round(time.time() - t0, 2)
    pct_nulos = None
    if df is not None:
        filas = df.count()
        cols = df.columns
        nulos = df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in cols]).collect()[0]
        pct_nulos = round(sum(v or 0 for v in nulos) / (filas * len(cols)) * 100, 4) if filas else 0.0
    fila = {"run_id": RUN_ID, "origen": ORIGEN, "timestamp": datetime.datetime.now().isoformat(timespec="seconds"),
            "notebook": NOTEBOOK, "etapa": etapa, "duracion_s": duracion, "filas": filas,
            "pct_nulos": pct_nulos, "memoria_mb": memoria_mb(), "cpu_pct": psutil.cpu_percent(interval=0.2),
            "formato": FORMATO}
    pd.DataFrame([fila]).to_csv(LOG_MON, mode="a", index=False, header=not os.path.exists(LOG_MON))
    print(f"⏱️ {etapa}: {duracion} s · filas={filas} · nulos={pct_nulos}% · mem={fila['memoria_mb']} MB")

def mostrar(df, n=20):
    """display() de Databricks no existe en Jupyter/Colab: se muestra como tabla pandas."""
    from IPython.display import display
    display(df.limit(n).toPandas() if hasattr(df, "toPandas") else df)

print(f"Notebook: {NOTEBOOK} · Entorno: {ORIGEN} · Formato de tablas: {FORMATO.upper()} · Spark {spark.version}")
print("LAKE =", LAKE, "| EVID =", EVID)

In [ ]:
bronce_cc = leer("bronce", "transacciones").drop("_archivo")
bronce_sfc = leer("bronce", "sfc_tarjetas").drop("_archivo")
bitacora = []

def paso(cod, tabla, desc, antes_df, despues_df):
    a, d = antes_df.count(), despues_df.count()
    bitacora.append({"codigo": cod, "tabla": tabla, "transformacion": desc,
                     "filas_antes": a, "filas_despues": d, "filas_eliminadas": a - d})
    print(f"{cod} · {tabla:<14} · {desc:<55} · {a:>9,} → {d:>9,} (−{a-d:,})")
    return despues_df.cache()

cc, sfc = bronce_cc.cache(), bronce_sfc.cache()

### T1 · Deduplicación
En F1 se eliminan filas idénticas en las 31 columnas de datos (se esperan 1.081). En F2, registros repetidos de la misma entidad, franquicia, fecha e indicador.

In [ ]:
t0 = time.time()
cols_datos = ["Time"] + [f"V{i}" for i in range(1, 29)] + ["Amount", "Class"]
cc = paso("T1", "transacciones", "Deduplicación sobre las 31 columnas de datos", cc, cc.dropDuplicates(cols_datos))
llave_sfc = ["codigoentidad", "fechacorte", "cod_uca", "subcuenta"]
sfc = paso("T1", "sfc_tarjetas", "Deduplicación por entidad-fecha-franquicia-indicador", sfc, sfc.dropDuplicates(llave_sfc))
registrar("T1_deduplicacion", t0)

### T2 · Descarte de nulos
No se imputa: un monto o una etiqueta inventada distorsionaría la tasa de fraude. Se descartan registros sin los campos críticos.

In [ ]:
t0 = time.time()
cc = paso("T2", "transacciones", "Descarte de filas con Time, Amount o Class nulos", cc,
          cc.dropna(subset=["Time", "Amount", "Class"]))
sfc = paso("T2", "sfc_tarjetas", "Descarte de filas sin entidad, fecha o valor", sfc,
           sfc.dropna(subset=["nombreentidad", "fechacorte", "total_tarjetas"]))
registrar("T2_nulos", t0)

### T3 · Tipificación y variables derivadas
- F1: `id` técnico de transacción, `dia` (1 o 2), `hora` del día (0–23), `log_amount` = ln(1 + monto) y `rango_monto`.
- F2: `fechacorte` → fecha, valores → enteros, `anio` y `mes`.

In [ ]:
t0 = time.time()
tip_cc = (cc.withColumn("id", F.monotonically_increasing_id())
            .withColumn("Class", F.col("Class").cast("int"))
            .withColumn("dia", (F.col("Time") / 86400).cast("int") + 1)
            .withColumn("hora", ((F.col("Time") / 3600) % 24).cast("int"))
            .withColumn("log_amount", F.log1p("Amount"))
            .withColumn("rango_monto",
                        F.when(F.col("Amount") < 10, "1) 0-10")
                         .when(F.col("Amount") < 50, "2) 10-50")
                         .when(F.col("Amount") < 200, "3) 50-200")
                         .when(F.col("Amount") < 1000, "4) 200-1000")
                         .otherwise("5) 1000+")))
cc = paso("T3", "transacciones", "Tipificación + id, día, hora, log_amount, rango_monto", cc, tip_cc)

fecha_txt = F.substring(F.col("fechacorte"), 1, 10)
tip_sfc = (sfc.withColumn("fecha", F.coalesce(F.to_date(fecha_txt, "yyyy-MM-dd"), F.to_date(F.col("fechacorte"), "dd/MM/yyyy")))
              .withColumn("subcuenta", F.col("subcuenta").cast("int"))
              .withColumn("persona_natural", F.col("persona_natural").cast("double").cast("long"))
              .withColumn("persona_juridica", F.col("persona_juridica").cast("double").cast("long"))
              .withColumn("valor", F.col("total_tarjetas").cast("double").cast("long"))
              .withColumn("anio", F.year("fecha")).withColumn("mes", F.month("fecha"))
              .dropna(subset=["fecha", "valor"]))
sfc = paso("T3", "sfc_tarjetas", "Tipificación de fecha y valores (descarta no convertibles)", sfc, tip_sfc)
registrar("T3_tipificacion", t0)

### T4 · Estandarización de categorías (F2)
Se normalizan mayúsculas y espacios, se unifica la franquicia (p. ej. `CREDIBANCO-VISA` → `VISA`) y la descripción larga se convierte en un código corto de indicador.

In [ ]:
t0 = time.time()
indicadores = {5: "tarjetas_vigentes", 10: "tarjetas_vigentes_mes", 15: "tarjetas_canceladas",
               20: "tarjetas_bloqueadas", 25: "tx_compras_nal", 30: "tx_avances_nal",
               35: "tx_compras_ext", 40: "tx_avances_ext"}
mapa_ind = F.create_map(*[x for k, v in indicadores.items() for x in (F.lit(k), F.lit(v))])
limpio = lambda c: F.upper(F.trim(F.regexp_replace(F.col(c), r"\s+", " ")))
franq = limpio("nombre_uca")
est_sfc = (sfc.withColumn("entidad", limpio("nombreentidad"))
              .withColumn("franquicia", F.when(franq.contains("VISA"), "VISA")
                                         .when(franq.contains("MASTER"), "MASTERCARD")
                                         .when(franq.contains("AMERICAN"), "AMERICAN EXPRESS")
                                         .when(franq.contains("DINERS"), "DINERS")
                                         .otherwise(franq))
              .withColumn("indicador", F.coalesce(mapa_ind[F.col("subcuenta")],
                                                  F.concat(F.lit("subcuenta_"), F.col("subcuenta").cast("string"))))
              .withColumn("producto", F.when(F.lower("descripcion").contains("débito") | F.lower("descripcion").contains("debito"),
                                             "TARJETA_DEBITO").otherwise("TARJETA_CREDITO")))
sfc = paso("T4", "sfc_tarjetas", "Estandarización de entidad, franquicia, indicador y producto", sfc, est_sfc)
print(sfc.groupBy("franquicia").count().orderBy(F.desc("count")).toPandas().head(10))
registrar("T4_estandarizacion", t0)

### T5 · Filtrado de atípicos
- **F1:** se eliminan montos negativos (imposibles). Los montos muy altos **no se eliminan** porque en fraude un atípico puede ser justamente el evento a detectar: se **marcan** con la regla IQR (`es_atipico_monto`).
- **F2:** se eliminan valores negativos.

In [ ]:
t0 = time.time()
q1, q3 = cc.approxQuantile("Amount", [0.25, 0.75], 0.001)
lim_sup = q3 + 1.5 * (q3 - q1)
print(f"IQR del monto: Q1={q1:.2f} Q3={q3:.2f} → límite superior {lim_sup:.2f} EUR")
cc = paso("T5", "transacciones", "Filtro de montos negativos + marca IQR de atípicos", cc,
          cc.filter(F.col("Amount") >= 0).withColumn("es_atipico_monto", (F.col("Amount") > lim_sup).cast("int")))
sfc = paso("T5", "sfc_tarjetas", "Filtro de valores negativos", sfc, sfc.filter(F.col("valor") >= 0))
atip = cc.groupBy("Class").agg(F.avg("es_atipico_monto").alias("pct_atipicos")).toPandas()
print(atip)
registrar("T5_atipicos", t0)

### Persistencia de plata (F1 y F2)

In [ ]:
t0 = time.time()
plata_cc = guardar(cc.withColumn("producto", F.lit("TARJETA_CREDITO")), "plata", "transacciones")
registrar("plata_transacciones", t0, df=plata_cc)

t0 = time.time()
plata_sfc = guardar(sfc.select("entidad", "codigoentidad", "franquicia", "producto", "indicador", "subcuenta",
                               "fecha", "anio", "mes", "valor", "persona_natural", "persona_juridica", "_fuente"),
                    "plata", "sfc_tarjetas")
registrar("plata_sfc_tarjetas", t0, df=plata_sfc)

### T6 · Unión de fuentes → `plata.exposicion`
Las dos fuentes **no comparten una llave de cliente** (F1 es europea y anonimizada; F2 es agregada por entidad colombiana). Se integran mediante una **dimensión conformada `producto = TARJETA_CREDITO`**: a las transacciones de compra con tarjeta de crédito reportadas por cada entidad a la SFC se les aplica la tasa de fraude y el monto medio de fraude observados en F1.

> Es un **escenario de referencia** (supuesto explícito: la tasa observada en F1 se aplica al mercado colombiano), útil para dimensionar la exposición, no una medición del fraude real en Colombia.

In [ ]:
t0 = time.time()
kpi_f1 = (plata_cc.groupBy("producto")
          .agg(F.avg("Class").alias("tasa_fraude"),
               F.avg(F.when(F.col("Class") == 1, F.col("Amount"))).alias("monto_medio_fraude_eur")))
tx_sfc = (plata_sfc.filter(F.col("indicador").isin("tx_compras_nal", "tx_compras_ext"))
          .groupBy("producto", "entidad", "fecha", "anio", "mes")
          .agg(F.sum(F.when(F.col("indicador") == "tx_compras_nal", F.col("valor")).otherwise(0)).alias("tx_compras_nal"),
               F.sum(F.when(F.col("indicador") == "tx_compras_ext", F.col("valor")).otherwise(0)).alias("tx_compras_ext"))
          .withColumn("tx_compras", F.col("tx_compras_nal") + F.col("tx_compras_ext")))
antes_union = tx_sfc.count()
exposicion = (tx_sfc.join(F.broadcast(kpi_f1), on="producto", how="inner")
              .withColumn("fraudes_estimados", F.round(F.col("tx_compras") * F.col("tasa_fraude"), 0))
              .withColumn("monto_expuesto_eur", F.round(F.col("fraudes_estimados") * F.col("monto_medio_fraude_eur"), 2)))
exposicion = paso("T6", "exposicion", "Unión SFC (transacciones) × F1 (tasa de fraude) por producto", tx_sfc, exposicion)
plata_exp = guardar(exposicion, "plata", "exposicion")
registrar("T6_union_fuentes", t0, df=plata_exp)
mostrar(plata_exp.orderBy(F.desc("fecha"), F.desc("tx_compras")), 10)

## Bitácora de transformaciones (evidencia A2.1)

In [ ]:
bit = pd.DataFrame(bitacora)
bit.to_csv(f"{EVID}/tablas/bitacora_limpieza.csv", index=False)
mostrar(bit)
print(f"F1: {bit[bit.tabla=='transacciones'].filas_antes.iloc[0]:,} filas en bronce → {plata_cc.count():,} en plata")

## Cierre
Respalda el Lakehouse (en Colab) y confirma la ejecución completa.

In [ ]:
respaldar_lake()
print("✅ Notebook", NOTEBOOK, "terminado · run_id:", RUN_ID)